In [34]:
import pandas as pd
import numpy as np
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
import statsmodels.api as sm

In [36]:
# Load your data
data = pd.read_csv('20241124 Gen.csv')

In [38]:
house_df = data[data['Category'] == 'house']
condo_df = data[data['Category'] == 'condo']
apt_df = data[data['Category'] == 'apartment']
com_df = data[data['Category'] == 'commercial']
land_df = data[data['Category'] == 'land']
RR_df = data[data['Property Type'] == 'RR']
CR_df = data[data['Property Type'] == 'CR']

In [40]:
def remove_outliers_iqr(df, feature):
    Q1 = df[feature].quantile(0.25)
    Q3 = df[feature].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    df_filtered = df[(df[feature] >= lower_bound) & (df[feature] <= upper_bound)]
    
    return df_filtered

def remove_outliers_iqr_list(df, feature_list):
    df_c = df.copy()
    for feature in feature_list:
        df_c = remove_outliers_iqr(df_c, feature)
        
    return df_c

### ET with House Data

In [10]:
# Data preparation
y = house_df['Price']
X = house_df[["LRTHubDist", 'Num_Bedrooms', 'Num_Bathrooms', 'Floor_Area', 'Land_Area', 
              'BusCount', 'SchoolCount', 
              'MallCount', 'HospitalCount']]

In [26]:
def objective(trial):
    # Hyperparameter ranges as specified
    params = {
        'n_estimators': trial.suggest_categorical('n_estimators', [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150, 160, 170, 180, 190, 200]),
        'max_depth': trial.suggest_int('max_depth', 2, 20),
        'max_features': trial.suggest_int('max_features', 2, 13),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 2, 10),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10)
    }
    
    # Create Extra Trees Regressor with current hyperparameters
    et_regressor = ExtraTreesRegressor(
        n_estimators=params['n_estimators'],
        max_depth=params['max_depth'],
        max_features=params['max_features'],
        min_samples_leaf=params['min_samples_leaf'],
        min_samples_split=params['min_samples_split'],
        criterion='squared_error',
        bootstrap=False
    )
    
    # Perform cross-validation
    scores = cross_val_score(et_regressor, X, y, cv=5, scoring='neg_mean_squared_error')
    return np.mean(-scores)

In [28]:
def optimize_extra_trees(X, y):
    # Create a study object and optimize the objective function
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=100)
    
    # Print best parameters and score
    print("Best trial:")
    trial = study.best_trial
    print(f"  Value: {trial.value}")
    print("  Params: ")
    for key, value in trial.params.items():
        print(f"    {key}: {value}")
    
    return study.best_params

In [30]:
# Assuming df is already loaded and preprocessed
y = house_df['Price']
X = house_df[["LRTHubDist", 'Num_Bedrooms', 'Num_Bathrooms', 'Floor_Area', 'Land_Area', 
        'BusCount', 'SchoolCount', 'MallCount', 'HospitalCount']]

# Optimize and get best parameters
best_params = optimize_extra_trees(X, y)

# Train final model with best parameters
final_model = ExtraTreesRegressor(**best_params, criterion='squared_error', bootstrap=False)
final_model.fit(X, y)

[I 2025-01-30 08:05:51,342] A new study created in memory with name: no-name-27fb1b09-a98d-4f49-9920-f50055f0dd1a
[I 2025-01-30 08:05:52,209] Trial 0 finished with value: 3.379121408974782e+16 and parameters: {'n_estimators': 10, 'max_depth': 12, 'max_features': 3, 'min_samples_leaf': 4, 'min_samples_split': 9}. Best is trial 0 with value: 3.379121408974782e+16.
[I 2025-01-30 08:05:57,103] Trial 1 finished with value: 3.3123026254974636e+16 and parameters: {'n_estimators': 30, 'max_depth': 11, 'max_features': 11, 'min_samples_leaf': 9, 'min_samples_split': 10}. Best is trial 1 with value: 3.3123026254974636e+16.
[I 2025-01-30 08:06:00,209] Trial 2 finished with value: 3.3965112767605736e+16 and parameters: {'n_estimators': 10, 'max_depth': 20, 'max_features': 11, 'min_samples_leaf': 5, 'min_samples_split': 10}. Best is trial 1 with value: 3.3123026254974636e+16.
[I 2025-01-30 08:06:06,359] Trial 3 finished with value: 3.333909999042183e+16 and parameters: {'n_estimators': 30, 'max_dept

Best trial:
  Value: 3.2180780727205324e+16
  Params: 
    n_estimators: 50
    max_depth: 8
    max_features: 10
    min_samples_leaf: 10
    min_samples_split: 3


ExtraTreesRegressor(max_depth=8, max_features=10, min_samples_leaf=10,
                    min_samples_split=3, n_estimators=50)